This script take `osrm/puma_neearest_college_driving_time.parquet`, merge with cps cleaned data and perform 2SLS without geographic controls

In [1]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from linearmodels.iv import IV2SLS

import os
os.chdir('/Users/ryotarohiraki/Desktop/Spring 2026/Capstone/projects')

/opt/anaconda3/envs/mosaiks-env/lib/python3.11/site-packages/statsmodels/tools/tools.py:6: UserWarning: A NumPy version >=1.26.4 and <2.7.0 is required for this version of SciPy (detected version 1.25.1)
  import scipy.linalg


In [ ]:
#retrieve data
cps14 = pd.read_parquet('dataset/cleaned_dataset/cleaned_cps14.parquet')
# cps13 = pd.read_parquet('dataset/cleaned_dataset/cleaned_cps13.parquet')
# cps12 = pd.read_parquet('dataset/cleaned_dataset/cleaned_cps12.parquet')


driving = pd.read_parquet('dataset/osrm/puma_nearest_college_driving_time.parquet')


Index(['STATE', 'PUMA', 'puma_rep_lon', 'puma_rep_lat', 'nearest_college_lon',
       'nearest_college_lat', 'driving_time_seconds', 'driving_time_status',
       'origin_snap_m', 'destination_snap_m', 'osrm_data_version', 'osrm_url'],
      dtype='object')

In [5]:
cps14.columns


Index(['person_num', 'log_wage', 'educ', 'exp', 'exp2', 'female', 'black',
       'mv', 'age', 'birth_place', 'employed', 'PUMA', 'STATE', 'STATE_PUMA',
       'same_home_as_14ys', 'same_home_as_13ys', 'same_home_as_12ys', 'mover',
       'age_bins', 'pums_weight'],
      dtype='object')

In [7]:
driving.columns

Index(['STATE', 'PUMA', 'puma_rep_lon', 'puma_rep_lat', 'nearest_college_lon',
       'nearest_college_lat', 'driving_time_seconds', 'driving_time_status',
       'origin_snap_m', 'destination_snap_m', 'osrm_data_version', 'osrm_url'],
      dtype='object')

In [8]:
driving.head()

,STATE,PUMA,puma_rep_lon,puma_rep_lat,nearest_college_lon,nearest_college_lat,driving_time_seconds,driving_time_status,origin_snap_m,destination_snap_m,osrm_data_version,osrm_url
0,01,00100,-87.787718,34.656356,-87.678089,34.739788,1824.8,ok,507.297121,33.522866,None,http://127.0.0.1:5050
1,01,00200,-87.032229,34.774908,-86.964698,34.806793,748.2,ok,218.365340,39.384468,None,http://127.0.0.1:5050
2,01,00300,-87.163510,34.551995,-86.949674,34.645844,1576.7,ok,118.078910,38.064535,None,http://127.0.0.1:5050
3,01,00401,-86.458264,34.798057,-86.568502,34.783368,1260.0,ok,27.999265,45.807524,None,http://127.0.0.1:5050
4,01,00402,-86.707100,34.689432,-86.752565,34.681465,1751.5,ok,12.557687,22.558689,None,http://127.0.0.1:5050


In [9]:
cps14.head()

,person_num,log_wage,educ,exp,exp2,female,black,mv,age,birth_place,employed,PUMA,STATE,STATE_PUMA,same_home_as_14ys,same_home_as_13ys,same_home_as_12ys,mover,age_bins,pums_weight
3802,4,16.659324,0.0,20.0,400.0,1,0,7.0,26,1,1,00501,01,01_00501,1,1,1,1,30-39,15
3847,3,16.979841,12.0,8.0,64.0,0,1,5.0,26,1,1,01801,01,01_01801,1,1,1,1,30-39,189
3889,2,15.806328,12.0,0.0,0.0,0,1,5.0,18,1,1,01404,01,01_01404,1,1,1,1,20-29,155
3896,1,16.964838,14.0,23.0,529.0,0,0,7.0,43,1,1,00100,01,01_00100,1,1,0,1,50-59,25
3897,2,17.585665,18.0,17.0,289.0,1,0,7.0,41,1,1,00100,01,01_00100,1,1,1,1,50-59,26


In [11]:
#merge two datasets by PUMA and STATE
df_merged = cps14.merge(driving, how="left", on=['PUMA', 'STATE'], validate='many_to_one')

In [13]:
cps14.shape

(147208, 20)

In [12]:
df_merged.shape

(147208, 30)

In [17]:
sum(df_merged.driving_time_seconds.isna())

1319

In [18]:
df_analysis = df_merged.dropna(
    subset=["driving_time_seconds"]
).copy()

In [19]:
#perform 2SLS
#baseline 2SLS

formula1 = "log_wage ~ 1 + exp + exp2 + [educ ~ driving_time_seconds]"
base = IV2SLS.from_formula(formula=formula1, data=df_analysis, weights=df_analysis["pums_weight"]).fit(
    cov_type="clustered",
    clusters=df_analysis["STATE_PUMA"]
)

formula2 = "log_wage ~ 1 + exp + exp2 + female + black + [educ ~ driving_time_seconds]"
base2 = IV2SLS.from_formula(formula=formula2, data=df_analysis, weights=df_analysis["pums_weight"]).fit(
    cov_type="clustered",
    clusters=df_analysis["STATE_PUMA"]
)

print(base.summary)
print(base.first_stage)
print(base2.summary)
print(base2.first_stage)

                          IV-2SLS Estimation Summary                          
Dep. Variable:               log_wage   R-squared:                      0.0651
Estimator:                    IV-2SLS   Adj. R-squared:                 0.0651
No. Observations:              145889   F-statistic:                    5444.8
Date:                Wed, Oct 07 2026   P-value (F-stat)                0.0000
Time:                        06:56:00   Distribution:                  chi2(3)
Cov. Estimator:             clustered                                         
                                                                              
                             Parameter Estimates                              
            Parameter  Std. Err.     T-stat    P-value    Lower CI    Upper CI
------------------------------------------------------------------------------
Intercept      14.221     0.1961     72.533     0.0000      13.837      14.605
exp            0.0292     0.0044     6.6910     0.00